In [1]:
import os
import qsprpred

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
os.makedirs("dataset_outputs/hERG/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="data/src/datasets/herg_large/data/hERG_large.csv",
    store_dir="dataset_outputs/CK1/data",
    name="CK1Dataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
)

In [3]:
dataset.getDF()

,Y,Drug,QSPRID,Y_original
QSPRID,,,,
CK1Dataset_00000,True,C=Cc1c(CNC23CCC(CCc4c(F)cnc5ccc(OC)nc45)(CC2)O...,CK1Dataset_00000,1
CK1Dataset_00001,True,C=Cc1c(CN[C@@]23CC[C@@](CCc4c(F)cnc5ccc(OC)nc4...,CK1Dataset_00001,1
CK1Dataset_00002,True,CC(C)(O)CCOc1ccc2ncc(F)c(CCC34CCC(NCc5ccc6c(n5...,CK1Dataset_00002,1
CK1Dataset_00003,True,CC(C)CCOc1ccc2ncc(F)c(CCC34CCC(NCc5ccc6c(n5)NC...,CK1Dataset_00003,1
CK1Dataset_00004,True,CCC(=O)[C@]1(C)Oc2ccc(CN[C@]34CC[C@](CCc5c(F)c...,CK1Dataset_00004,1
...,...,...,...,...
CK1Dataset_14317,True,COc1cc(-c2nc3n(n2)CCOC3c2ccccc2C(F)(F)F)ccc1-n...,CK1Dataset_14317,1
CK1Dataset_14318,True,COc1cc(-c2nc3n(n2)CCC[C@H]3c2ccc(F)cc2C)ccc1-n...,CK1Dataset_14318,1
CK1Dataset_14319,True,COc1cc(-c2nc3n(n2)CCC[C@@H]3c2ccc(F)cc2C)ccc1-...,CK1Dataset_14319,1


In [4]:
dataset.prepareDataset(
    split=RandomSplit(test_fraction=0.2, dataset=dataset),
    feature_calculators=[MorganFP(radius=3, nBits=2048)],
    recalculate_features=True,
)

In [5]:
from qsprpred.data.descriptors.sets import RDKitDescs

rdkit_descs = RDKitDescs()

dataset.addDescriptors([rdkit_descs])

dataset.descriptorSets

In [6]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [7]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import pandas as pd
def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    return my_df

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = dataset.X
X1, X2, y1, y2 = train_test_split(dataset.X, dataset.y, test_size=0.25, random_state=42)
X3 = dataset.X_ind
y3 = dataset.y_ind

imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)

scaler = StandardScaler()
X1 = scaler.fit_transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)



In [9]:
dataset.getDF()["Y"].count()


14322

In [10]:
test_par = {'weight_decay': 0.0001, 'patience': 50, 
            'neuron_layers': [5000, 2500],
            'n_epochs': 300, 'dropout_frac': 0.4,
            'act_fun': F.selu}
model_sts_3 = STFullyConnected(n_dim=X1.shape[1],  # počet vstupních neuronů (počet deskriptorů)
    n_class=1,  # regresní úloha (1 výstup)
    gpus=[],
    device="cuda",
    batch_size=256,is_reg=False, **test_par)
model_sts_3.fit(X1, y1)
res = model_sts_3.predict(X2)
res = res >0.5
print(f1_score(res, y2))

0.8467950560505892


In [11]:
from torch import optim
import torch
my_dict_ult = {
    "act_fun": [F.selu],
    "dropout_frac": [0.1, 0.4, 0.5, 0.6],
    "patience": [75],
    "tol": [1e-5],
    "weight_decay": [1e-4, 1e-3, 1e-5],
    "n_epochs": [200,300, 400],
    "neuron_layers": [[ 5000, 2500], [ 1024, 512, 256, 128, 64, 32, 16, 8, 4],  [2048, 1024, 512, 256, 128], [4096, 2048], [7500, 3750]]
    , "batch_size": [256]
    , "optimizer": [optim.AdamW] 
}
df_batch_ult = test_fun(my_dict_ult, X1, y1, X2, y2)

Device used: cuda
1 / 180
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 256, 'dropout_frac': 0.1, 'n_epochs': 200, 'neuron_layers': [5000, 2500], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.8488505747126437
0.8164048865619546
2 / 180
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 256, 'dropout_frac': 0.1, 'n_epochs': 200, 'neuron_layers': [5000, 2500], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.001}
0.8488505747126437
0.8164048865619546
3 / 180


KeyboardInterrupt: 

In [ ]:
#{'act_fun': <function selu at 0x7f5daf74b060>, 'batch_size': 256, 'dropout_frac': 0.1, 'n_epochs': 300,
# 'neuron_layers': [2048, 1024, 512, 256, 128], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}

In [14]:
from torch import optim
import torch
my_dict_ult_2 = {
    "act_fun": [F.selu],
    "dropout_frac": [0.35, 0.45, 0.4],
    "patience": [75],
    "tol": [1e-5],
    "weight_decay": [1e-4],
    "n_epochs": [300, 450, 350],
    "neuron_layers": [ [4096, 2048], [7500, 3750]]
    , "batch_size": [256, 128]
    , "optimizer": [optim.AdamW] 
}
df_batch_ult_2 = test_fun(my_dict_ult_2, X1, y1, X2, y2)

Device used: cuda
1 / 36
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 256, 'dropout_frac': 0.35, 'n_epochs': 300, 'neuron_layers': [4096, 2048], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.8483630097645032
0.8157068062827225
2 / 36
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 256, 'dropout_frac': 0.35, 'n_epochs': 300, 'neuron_layers': [7500, 3750], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.8481012658227848
0.8157068062827225
3 / 36
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 256, 'dropout_frac': 0.35, 'n_epochs': 450, 'neuron_layers': [4096, 2048], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.8476327116212339
0.8146596858638744
4 / 36
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 256, 'dropout_frac': 0.35, 'n_epochs': 450, 'neuron_layers': [750

In [15]:
df_batch_ult_2.sort_values(by="F1", ascending=False, inplace=True)
df_batch_ult_2.to_csv('qsprpred/extra/gpu/models/dataset_outputs/hERG/tabs/hergbatch3.csv')

In [17]:
from torch import optim
import torch
my_dict_ult_3 = {
    "act_fun": [F.selu, F.gelu, F.elu, F.silu],
    "dropout_frac": [ 0.45, 0.4, 0.5],
    "patience": [75],
    "tol": [1e-5],
    "weight_decay": [1e-4],
    "n_epochs": [300, 450, 350],
    "neuron_layers": [  [7500, 3750]]
    , "batch_size": [512, 256, 128]
    , "optimizer": [optim.AdamW] 
}
df_batch_ult_3 = test_fun(my_dict_ult_3, X1, y1, X2, y2)

Device used: cuda
1 / 108
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 512, 'dropout_frac': 0.45, 'n_epochs': 300, 'neuron_layers': [7500, 3750], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.8469914040114613
0.8136125654450261
2 / 108
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 512, 'dropout_frac': 0.45, 'n_epochs': 450, 'neuron_layers': [7500, 3750], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.8506605399195865
0.8184991273996509
3 / 108
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 512, 'dropout_frac': 0.45, 'n_epochs': 350, 'neuron_layers': [7500, 3750], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.8487106017191977
0.8157068062827225
4 / 108
{'act_fun': <function selu at 0x7ffb5a5c7420>, 'batch_size': 512, 'dropout_frac': 0.4, 'n_epochs': 300, 'neuron_layers': [

In [18]:
df_batch_ult_3.sort_values(by="F1", ascending=False, inplace=True)
df_batch_ult_3.to_csv('qsprpred/extra/gpu/models/dataset_outputs/hERG/tabs/hergbatch4.csv')

In [20]:
df_batch_ult_3

,act_fun,batch_size,dropout_frac,n_epochs,neuron_layers,optimizer,patience,tol,weight_decay,F1,Acc
49,<built-in function gelu>,128,0.40,450,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.883721,0.856894
46,<built-in function gelu>,128,0.45,450,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.883404,0.856545
52,<built-in function gelu>,128,0.50,450,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.883404,0.856545
47,<built-in function gelu>,128,0.45,350,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.882587,0.855497
53,<built-in function gelu>,128,0.50,350,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.882270,0.855148
...,...,...,...,...,...,...,...,...,...,...,...
25,<function selu at 0x7ffb5a5c7420>,128,0.50,450,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.847106,0.814660
23,<function selu at 0x7ffb5a5c7420>,128,0.40,350,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.847039,0.814311
19,<function selu at 0x7ffb5a5c7420>,128,0.45,450,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.847018,0.814660
0,<function selu at 0x7ffb5a5c7420>,512,0.45,300,"[7500, 3750]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.846991,0.813613
